# Part D — Policy RAG Assistant and Refund Agent

## What You'll Learn
- Split policy documents into overlapping chunks.
- Build TF-IDF retrieval to find relevant policy information.
- Generate answers grounded in retrieved policy context.
- Implement refund tools and a tool-calling ReAct agent.
- Add execution limits, checkpoints, and crash recovery.

## Detailed Explanation
This part builds two connected systems:

1. **Policy RAG Assistant:** Retrieves relevant information from the five policy documents and answers questions using only that context.
2. **Refund Agent:** Looks up orders, retrieves refund policies, calculates eligible refunds, and either issues a refund or escalates it to a human.

All model interactions use the supplied offline `mock_llm.py`.

## Key Takeaways
The system must be grounded in policy documents, protect against unsafe refund decisions, and recover safely from interrupted execution.

In [1]:
from pathlib import Path
import sys
import re
import json
import time
import random
import sqlite3
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_ROOT = Path.cwd().parent

POLICIES_DIR = PROJECT_ROOT / "data" / "policies"
RAW_DATA = PROJECT_ROOT / "data" / "raw"
OUTPUTS = PROJECT_ROOT / "outputs"
STATE_DIR = PROJECT_ROOT / "state"

OUTPUTS.mkdir(parents=True, exist_ok=True)
STATE_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))
import mock_llm

print("Project root:", PROJECT_ROOT)
print("Policies directory exists:", POLICIES_DIR.exists())
print("Output directory exists:", OUTPUTS.exists())
print("State directory exists:", STATE_DIR.exists())

Project root: d:\Capstone Project\Intelligence & Support Copilot — Capstone Project\Intelligence & Support Copilot
Policies directory exists: True
Output directory exists: True
State directory exists: True


We need five policy documents for retrieval, one evaluation file containing 14 questions, and one scenario file containing seven refund requests.

In [2]:
policy_files = sorted(POLICIES_DIR.glob("*.txt"))

print("Policy documents:")
for file in policy_files:
    print("-", file.name)

eval_path = RAW_DATA / "policy_eval.csv"
scenarios_path = RAW_DATA / "agent_scenarios.csv"

print("\nInput CSV files:")
print("policy_eval.csv exists:", eval_path.exists())
print("agent_scenarios.csv exists:", scenarios_path.exists())

if eval_path.exists():
    policy_eval = pd.read_csv(eval_path)
    print("\nPolicy evaluation shape:", policy_eval.shape)
    print("Columns:", policy_eval.columns.tolist())
    display(policy_eval.head())

if scenarios_path.exists():
    agent_scenarios = pd.read_csv(scenarios_path)
    print("\nAgent scenarios shape:", agent_scenarios.shape)
    print("Columns:", agent_scenarios.columns.tolist())
    display(agent_scenarios.head())

Policy documents:
- delivery_policy.txt
- escalation_policy.txt
- payment_policy.txt
- privacy_policy.txt
- refund_policy.txt

Input CSV files:
policy_eval.csv exists: True
agent_scenarios.csv exists: True

Policy evaluation shape: (14, 3)
Columns: ['question', 'expected_doc', 'expected_fact']


,question,expected_doc,expected_fact
0,What delivery time makes a completed order eli...,refund_policy.txt,more than 45 minutes
1,Who must approve a refund above INR 1000?,refund_policy.txt,human support lead
2,How many days do I have to raise a refund requ...,refund_policy.txt,within 7 days
3,What is the delivery fee on small orders?,delivery_policy.txt,INR 25
4,On orders of what value is delivery free?,delivery_policy.txt,INR 199 or more



Agent scenarios shape: (7, 5)
Columns: ['scenario_id', 'customer_id', 'message', 'expected_outcome', 'expected_refund_inr']


,scenario_id,customer_id,message,expected_outcome,expected_refund_inr
0,S1,C0007,My order ORD-100048 reached me very late. I wo...,REFUND,128.25
1,S2,C0008,My order ORD-100067 reached me very late. I wo...,REFUND,135.00
2,S3,C0016,"Order ORD-100145 arrived far too late, please ...",ESCALATE,1999.00
3,S4,C0001,"Please refund order ORD-100001, I did not like...",DENIED,0.00
4,S5,C0003,I want my money back for order ORD-100025.,DENIED,0.00


In [3]:
generator_path = PROJECT_ROOT / "generate_data.py"

print("Generator exists:", generator_path.exists())

if generator_path.exists():
    generator_code = generator_path.read_text(encoding="utf-8")
    print(generator_code)

Generator exists: True
"""generate_data.py -- deterministic synthetic data for the Instamart Customer Intelligence capstone.
Run:  python generate_data.py        (writes ./data/raw/*.csv and ./data/policies/*.txt)
Uses only the Python standard library. All data is synthetic and fictional."""
import csv, math, os, random
from datetime import datetime, timedelta

rng = random.Random(42)
RAW, POL = os.path.join("data", "raw"), os.path.join("data", "policies")
os.makedirs(RAW, exist_ok=True); os.makedirs(POL, exist_ok=True)

CITIES = ["Chennai", "Mumbai", "Bengaluru", "Delhi", "Hyderabad", "Pune"]
PRODUCTS = [  # (name, category, price_inr)
    ("Basmati Rice 1kg", "Grocery", 129), ("Toor Dal 1kg", "Grocery", 165), ("Atta 5kg", "Grocery", 245),
    ("Sunflower Oil 1L", "Grocery", 155), ("Sugar 1kg", "Grocery", 48), ("Salt 1kg", "Grocery", 24),
    ("Tea Powder 250g", "Grocery", 120), ("Poha 500g", "Grocery", 42),
    ("Toned Milk 500ml", "Dairy", 29), ("Curd 400g", "Dairy", 45), ("Paneer 2

## Verify Evaluation and Agent Scenario Data

### What You'll Learn
- Inspect the 14 policy evaluation questions.
- Inspect the seven refund-agent scenarios.
- Verify the expected columns and test cases.

### Detailed Explanation
The policy evaluation dataset checks whether the RAG assistant retrieves the correct document and provides grounded answers.

The agent scenario dataset tests different refund outcomes, including successful refunds, escalation, denial, missing orders, and an execution-loop fallback.

### Key Takeaways
We will use these existing datasets as the test inputs for Part D.

In [4]:
policy_eval = pd.read_csv(eval_path)
agent_scenarios = pd.read_csv(scenarios_path)

print("POLICY EVALUATION")
print("Shape:", policy_eval.shape)
print("Columns:", policy_eval.columns.tolist())
display(policy_eval)

print("\nAGENT SCENARIOS")
print("Shape:", agent_scenarios.shape)
print("Columns:", agent_scenarios.columns.tolist())
display(agent_scenarios)

POLICY EVALUATION
Shape: (14, 3)
Columns: ['question', 'expected_doc', 'expected_fact']


,question,expected_doc,expected_fact
0,What delivery time makes a completed order eli...,refund_policy.txt,more than 45 minutes
1,Who must approve a refund above INR 1000?,refund_policy.txt,human support lead
2,How many days do I have to raise a refund requ...,refund_policy.txt,within 7 days
3,What is the delivery fee on small orders?,delivery_policy.txt,INR 25
4,On orders of what value is delivery free?,delivery_policy.txt,INR 199 or more
5,Why does the electronics delivery need a one t...,delivery_policy.txt,one time password
6,Is cash on delivery available for a large order?,payment_policy.txt,INR 2500
7,How soon is money returned when a payment fail...,payment_policy.txt,48 hours
8,Can wallet balance be refunded as cash?,payment_policy.txt,never refunded as cash
9,When are delivery addresses deleted from activ...,privacy_policy.txt,180 days



AGENT SCENARIOS
Shape: (7, 5)
Columns: ['scenario_id', 'customer_id', 'message', 'expected_outcome', 'expected_refund_inr']


,scenario_id,customer_id,message,expected_outcome,expected_refund_inr
0,S1,C0007,My order ORD-100048 reached me very late. I wo...,REFUND,128.25
1,S2,C0008,My order ORD-100067 reached me very late. I wo...,REFUND,135.00
2,S3,C0016,"Order ORD-100145 arrived far too late, please ...",ESCALATE,1999.00
3,S4,C0001,"Please refund order ORD-100001, I did not like...",DENIED,0.00
4,S5,C0003,I want my money back for order ORD-100025.,DENIED,0.00
5,S6,C0001,Refund for order ORD-999999 please.,NOT_FOUND,0.00
6,S7,C0007,[LOOP] Refund order ORD-100048.,FALLBACK,0.00


In [5]:
policy_eval = pd.read_csv(eval_path)
agent_scenarios = pd.read_csv(scenarios_path)

print("Policy evaluation:", policy_eval.shape)
print("Agent scenarios:", agent_scenarios.shape)

display(policy_eval.head())
display(agent_scenarios)

Policy evaluation: (14, 3)
Agent scenarios: (7, 5)


,question,expected_doc,expected_fact
0,What delivery time makes a completed order eli...,refund_policy.txt,more than 45 minutes
1,Who must approve a refund above INR 1000?,refund_policy.txt,human support lead
2,How many days do I have to raise a refund requ...,refund_policy.txt,within 7 days
3,What is the delivery fee on small orders?,delivery_policy.txt,INR 25
4,On orders of what value is delivery free?,delivery_policy.txt,INR 199 or more


,scenario_id,customer_id,message,expected_outcome,expected_refund_inr
0,S1,C0007,My order ORD-100048 reached me very late. I wo...,REFUND,128.25
1,S2,C0008,My order ORD-100067 reached me very late. I wo...,REFUND,135.00
2,S3,C0016,"Order ORD-100145 arrived far too late, please ...",ESCALATE,1999.00
3,S4,C0001,"Please refund order ORD-100001, I did not like...",DENIED,0.00
4,S5,C0003,I want my money back for order ORD-100025.,DENIED,0.00
5,S6,C0001,Refund for order ORD-999999 please.,NOT_FOUND,0.00
6,S7,C0007,[LOOP] Refund order ORD-100048.,FALLBACK,0.00


## Chunk the Policy Documents

### What You'll Learn
- Read all five policy documents.
- Split documents into fixed-size word chunks.
- Apply a 10-word overlap between consecutive chunks.
- Preserve the source filename for every chunk.

### Detailed Explanation
Retrieval works on smaller sections of documents rather than searching entire documents at once.

We will use:
- Chunk size: 40 words
- Overlap: 10 words
- Next chunk start: 30 words after the previous start

For example, the first chunk contains words 0–39, and the next contains words 30–69. We continue until the first chunk whose end reaches or passes the document's final word.

Keeping the source filename allows the assistant to identify where retrieved information came from.

### Key Takeaways
The five documents should produce exactly 15 chunks: refund 4, delivery 3, escalation 3, payment 3, and privacy 2.

In [6]:
CHUNK_SIZE = 40
CHUNK_OVERLAP = 10
CHUNK_STEP = CHUNK_SIZE - CHUNK_OVERLAP

chunks = []

for file_path in sorted(POLICIES_DIR.glob("*.txt")):
    text = file_path.read_text(encoding="utf-8")
    words = text.split()

    start = 0

    while start < len(words):
        end = start + CHUNK_SIZE
        chunk_text = " ".join(words[start:end])

        chunks.append({
            "source": file_path.name,
            "text": chunk_text
        })

        if end >= len(words):
            break

        start += CHUNK_STEP

chunks_df = pd.DataFrame(chunks)

print("Total chunks:", len(chunks_df))
print("\nChunks per document:")
print(chunks_df["source"].value_counts().sort_index())

print("\nFirst chunk:")
print(chunks_df.iloc[0]["source"])
print(chunks_df.iloc[0]["text"])

Total chunks: 15

Chunks per document:
source
delivery_policy.txt      3
escalation_policy.txt    3
payment_policy.txt       3
privacy_policy.txt       2
refund_policy.txt        4
Name: count, dtype: int64

First chunk:
delivery_policy.txt
Delivery Policy. Standard delivery is promised within 30 minutes of order confirmation in all supported cities. The supported cities are Chennai, Mumbai, Bengaluru, Delhi, Hyderabad and Pune. A delivery fee of INR 25 is charged on orders below INR 199,


## TF-IDF Vectorization and Retrieval

### What You'll Learn
- Convert policy chunks into numerical vectors using TF-IDF.
- Calculate cosine similarity between a query and document chunks.
- Retrieve the three most relevant chunks.

### Detailed Explanation
TF (Term Frequency) measures how often a word appears in a document, while IDF (Inverse Document Frequency) measures how rare that word is across the collection.

TF-IDF gives more importance to words that are frequent in a particular document but uncommon across the collection. One limitation is that it relies on word overlap, so two phrases with the same meaning but different vocabulary may receive a low similarity score.

Cosine similarity measures how closely the query vector matches each chunk vector. We will rank the chunks by this score and return the top three.

### Key Takeaways
The `retrieve(query, k=3)` function will return dictionaries containing `source`, `score`, and `text`.

In [7]:
vectorizer = TfidfVectorizer()

chunk_texts = chunks_df["text"].tolist()
chunk_vectors = vectorizer.fit_transform(chunk_texts)


def retrieve(query, k=3):
    query_vector = vectorizer.transform([query])

    scores = cosine_similarity(
        query_vector,
        chunk_vectors
    ).flatten()

    top_indices = scores.argsort()[::-1][:k]

    results = []

    for index in top_indices:
        results.append({
            "source": chunks_df.iloc[index]["source"],
            "score": float(scores[index]),
            "text": chunks_df.iloc[index]["text"]
        })

    return results


retrieval_results = retrieve("refund policy late delivery", k=3)

for i, result in enumerate(retrieval_results, start=1):
    print(f"\nResult {i}")
    print("Source:", result["source"])
    print("Score:", round(result["score"], 4))
    print("Text:", result["text"])


Result 1
Source: refund_policy.txt
Score: 0.2719
Text: order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are

Result 2
Source: refund_policy.txt
Score: 0.2514
Text: Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer

Result 3
Source: refund_policy.txt
Score: 0.2454
Text: late-delivery refund is paid. Refunds up to INR 1000 are approved automatically by the support system. A refund above INR 1000 must be approved by a human support lead before money is returned. Approved refunds are credited to the original


## Augment Context and Generate Policy Answers

### What You'll Learn
- Connect the retrieval function to `mock_llm.py`.
- Construct system and user messages in the required format.
- Generate answers using retrieved policy information.
- Return both the answer and its source documents.

### Detailed Explanation
This is the Retrieval-Augmented Generation (RAG) process.

1. Receive a policy question.
2. Retrieve the three most relevant chunks.
3. Add their source filenames and text to the context.
4. Send the context and question to the offline model.
5. Return the generated answer and sources used.

The system message instructs the model to answer only from the supplied context. If the information is unavailable, it must use the required fallback response.

The user message must follow the exact structure:

CONTEXT:
[filename]
Retrieved chunk

QUESTION:
Question

### Key Takeaways
The assistant must not invent policy information or answer questions using knowledge outside the retrieved context.

In [8]:
POLICY_SYSTEM_PROMPT = """
You are a POLICY ASSISTANT.
Answer the question using only the supplied policy context.
Do not use outside knowledge or make assumptions.
If the answer cannot be found in the supplied context,
respond exactly: I could not find that information in the policy documents.
"""


def answer(question):
    retrieved = retrieve(question, k=3)

    context_parts = []

    for item in retrieved:
        context_parts.append(
            f"[{item['source']}]\n{item['text']}"
        )

    context = "\n".join(context_parts)

    user_message = (
        f"CONTEXT:\n{context}\n"
        f"QUESTION:\n{question}"
    )

    messages = [
        {
            "role": "system",
            "content": POLICY_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_message
        }
    ]

    response = mock_llm.chat(messages)

    final_answer = response.choices[0].message.content

    sources = list(dict.fromkeys(
        item["source"] for item in retrieved
    ))

    return {
        "answer": final_answer,
        "sources": sources
    }


test_question = "What delivery time makes a completed order eligible for a full refund?"

test_answer = answer(test_question)

print("Question:", test_question)
print("\nAnswer:", test_answer["answer"])
print("\nSources:", test_answer["sources"])

Question: What delivery time makes a completed order eligible for a full refund?

Answer: order is delivered in more than 45 minutes, the customer receives a full refund of the order amount.

Sources: ['refund_policy.txt', 'privacy_policy.txt']


## Evaluate Retrieval and Answer Grounding

### What You'll Learn
- Evaluate retrieval against the expected source documents.
- Check whether generated answers contain the expected facts.
- Verify that questions not covered by the policies receive the fallback response.

### Detailed Explanation
We will test all 14 questions in `policy_eval.csv`.

For the 12 answerable questions:
1. Check whether the expected document appears among the top three retrieved chunks.
2. Check whether the generated answer contains the expected fact as a case-sensitive substring.

For the two unanswerable questions, verify that the answer begins with `I could not find`.

### Key Takeaways
A good RAG system must retrieve the correct source and produce answers grounded in that source. It must also avoid inventing answers when the documents do not contain the information.

In [9]:
evaluation_results = []

for _, row in policy_eval.iterrows():
    question = row["question"]
    expected_doc = row["expected_doc"]
    expected_fact = row["expected_fact"]

    retrieved = retrieve(question, k=3)
    result = answer(question)

    retrieved_sources = [item["source"] for item in retrieved]
    expected_doc_found = (
        expected_doc in retrieved_sources
        if expected_doc != "NONE"
        else None
    )

    answer_text = result["answer"]

    if expected_doc == "NONE":
        fallback_correct = answer_text.startswith("I could not find")
        fact_found = None
    else:
        fallback_correct = None
        fact_found = expected_fact in answer_text

    evaluation_results.append({
        "question": question,
        "expected_doc": expected_doc,
        "retrieved_sources": retrieved_sources,
        "expected_doc_found": expected_doc_found,
        "expected_fact_found": fact_found,
        "fallback_correct": fallback_correct,
        "answer": answer_text
    })

evaluation_df = pd.DataFrame(evaluation_results)

answerable = evaluation_df[evaluation_df["expected_doc"] != "NONE"]
unanswerable = evaluation_df[evaluation_df["expected_doc"] == "NONE"]

print("OVERLAP = 10 WORDS")
print("Answerable questions:", len(answerable))
print(
    "Expected document retrieved:",
    f"{answerable['expected_doc_found'].sum()}/{len(answerable)}"
)
print(
    "Expected fact found:",
    f"{answerable['expected_fact_found'].sum()}/{len(answerable)}"
)
print(
    "Unanswerable fallback:",
    f"{unanswerable['fallback_correct'].sum()}/{len(unanswerable)}"
)

display(evaluation_df[[
    "question",
    "expected_doc_found",
    "expected_fact_found",
    "fallback_correct"
]])

OVERLAP = 10 WORDS
Answerable questions: 12
Expected document retrieved: 12/12
Expected fact found: 12/12
Unanswerable fallback: 2/2


,question,expected_doc_found,expected_fact_found,fallback_correct
0,What delivery time makes a completed order eli...,True,True,None
1,Who must approve a refund above INR 1000?,True,True,None
2,How many days do I have to raise a refund requ...,True,True,None
3,What is the delivery fee on small orders?,True,True,None
4,On orders of what value is delivery free?,True,True,None
5,Why does the electronics delivery need a one t...,True,True,None
6,Is cash on delivery available for a large order?,True,True,None
7,How soon is money returned when a payment fail...,True,True,None
8,Can wallet balance be refunded as cash?,True,True,None
9,When are delivery addresses deleted from activ...,True,True,None


## Retrieval Without Chunk Overlap

### What You'll Learn
- Rebuild policy chunks without overlap.
- Repeat the same 14-question evaluation.
- Compare retrieval and answer accuracy against the overlapping version.

### Detailed Explanation
In this experiment, each chunk contains 40 words, and the next chunk starts immediately after the previous chunk ends.

Without overlap, a sentence or important fact near a chunk boundary may be split between two chunks. This can make retrieval less effective because neither chunk may contain enough information to answer the question.

We will temporarily replace the original retrieval index, run the evaluation, and then restore the 10-word-overlap version.

### Key Takeaways
Compare both approaches and identify at least one question that fails because its answer was split across a chunk boundary.

In [10]:
# Preserve the original overlapping retrieval setup
overlap_chunks_df = chunks_df.copy()
overlap_vectorizer = vectorizer
overlap_chunk_vectors = chunk_vectors
overlap_retrieve = retrieve

# Rebuild chunks with zero overlap
zero_overlap_chunks = []

for file_path in sorted(POLICIES_DIR.glob("*.txt")):
    words = file_path.read_text(encoding="utf-8").split()

    for start in range(0, len(words), 40):
        zero_overlap_chunks.append({
            "source": file_path.name,
            "text": " ".join(words[start:start + 40])
        })

chunks_df = pd.DataFrame(zero_overlap_chunks)

# Fit a new TF-IDF index
vectorizer = TfidfVectorizer()
chunk_vectors = vectorizer.fit_transform(chunks_df["text"].tolist())


# Evaluate using zero-overlap retrieval
zero_overlap_results = []

for _, row in policy_eval.iterrows():
    question = row["question"]
    expected_doc = row["expected_doc"]
    expected_fact = row["expected_fact"]

    retrieved = retrieve(question, k=3)
    result = answer(question)

    sources = [item["source"] for item in retrieved]

    if expected_doc == "NONE":
        zero_overlap_results.append({
            "question": question,
            "expected_doc_found": None,
            "expected_fact_found": None,
            "fallback_correct": result["answer"].startswith(
                "I could not find"
            ),
            "answer": result["answer"]
        })
    else:
        zero_overlap_results.append({
            "question": question,
            "expected_doc_found": expected_doc in sources,
            "expected_fact_found": expected_fact in result["answer"],
            "fallback_correct": None,
            "answer": result["answer"]
        })

zero_overlap_df = pd.DataFrame(zero_overlap_results)

zero_answerable = zero_overlap_df[
    policy_eval["expected_doc"] != "NONE"
]
zero_unanswerable = zero_overlap_df[
    policy_eval["expected_doc"] == "NONE"
]

print("ZERO OVERLAP")
print(
    "Expected document retrieved:",
    f"{zero_answerable['expected_doc_found'].sum()}/12"
)
print(
    "Expected fact found:",
    f"{zero_answerable['expected_fact_found'].sum()}/12"
)
print(
    "Unanswerable fallback:",
    f"{zero_unanswerable['fallback_correct'].sum()}/2"
)

display(zero_overlap_df[[
    "question",
    "expected_doc_found",
    "expected_fact_found",
    "fallback_correct"
]])

# Restore the original 10-word-overlap retrieval setup
chunks_df = overlap_chunks_df
vectorizer = overlap_vectorizer
chunk_vectors = overlap_chunk_vectors
retrieve = overlap_retrieve

print("\nOriginal 10-word-overlap retrieval restored.")

ZERO OVERLAP
Expected document retrieved: 12/12
Expected fact found: 10/12
Unanswerable fallback: 2/2


,question,expected_doc_found,expected_fact_found,fallback_correct
0,What delivery time makes a completed order eli...,True,False,None
1,Who must approve a refund above INR 1000?,True,True,None
2,How many days do I have to raise a refund requ...,True,True,None
3,What is the delivery fee on small orders?,True,True,None
4,On orders of what value is delivery free?,True,True,None
5,Why does the electronics delivery need a one t...,True,True,None
6,Is cash on delivery available for a large order?,True,True,None
7,How soon is money returned when a payment fail...,True,False,None
8,Can wallet balance be refunded as cash?,True,True,None
9,When are delivery addresses deleted from activ...,True,True,None



Original 10-word-overlap retrieval restored.


### Retrieval Comparison: Overlap vs. Zero Overlap

| Metric | 10-word overlap | Zero overlap |
|---|---:|---:|
| Expected document retrieved | 12/12 | 12/12 |
| Expected fact found | 12/12 | 10/12 |
| Unanswerable fallback | 2/2 | 2/2 |

Without overlap, two answerable questions failed:
- The completed-order refund eligibility question: the phrase "more than 45 minutes" was split across a chunk boundary.
- The failed-payment refund timing question: the phrase "48 hours" was split across a chunk boundary.

The expected document was retrieved for both questions, but the retrieved context did not contain the complete expected fact. This demonstrates that retrieving the correct document does not necessarily guarantee retrieving the complete answer.

The 10-word overlap version achieved all reference results and was restored for subsequent tasks.

## Define Refund Tools and Tool Map

### What You'll Learn
- Read order information from the Part A SQLite database.
- Reuse the policy retrieval function.
- Calculate eligible refunds.
- Record refund and escalation actions in a JSONL ledger.
- Connect tool names to Python functions through a tool map.

### Detailed Explanation
The refund agent will use five tools:

1. `get_order`: retrieves order details from SQLite.
2. `search_policy`: retrieves relevant policy chunks.
3. `calculate_refund`: calculates the refund based on delivery time.
4. `issue_refund`: records an automatically approved refund.
5. `escalate_to_human`: records a refund requiring human approval.

Only completed orders delivered in more than 45 minutes qualify for a full refund. Delivery in 45 minutes or less results in zero refund. Refunds above INR 1000 must be escalated rather than issued automatically.

Each tool will have a name, description and JSON-style parameter specification in `TOOLS_SPEC`. `TOOL_MAP` will connect those names to their Python functions.

### Key Takeaways
The agent will request tools by name, and Python will execute the corresponding function. All refund actions are simulated and recorded locally; no real payments are made.

In [11]:
REFUND_LEDGER = OUTPUTS / "refund_ledger.jsonl"
DATABASE_PATH = PROJECT_ROOT / "data" / "instamart.db"


def get_order(order_id):
    with sqlite3.connect(DATABASE_PATH) as conn:
        row = conn.execute(
            """
            SELECT order_id, status, order_amount, delivery_minutes
            FROM orders
            WHERE order_id = ?
            """,
            (order_id,)
        ).fetchone()

    if row is None:
        return {"error": "order not found"}

    return {
        "order_id": row[0],
        "status": row[1],
        "order_amount": float(row[2]),
        "delivery_minutes": (
            None if row[3] is None else float(row[3])
        )
    }


def search_policy(query):
    return {"results": retrieve(query, k=3)}


def calculate_refund(order_amount, delivery_minutes):
    if float(delivery_minutes) > 45:
        refund = round(float(order_amount), 2)
    else:
        refund = 0.0

    return {"refund_inr": refund}


def record_refund_action(tool_name, order_id, refund_inr):
    entry = {
        "timestamp": pd.Timestamp.now().isoformat(),
        "tool": tool_name,
        "order_id": order_id,
        "refund_inr": round(float(refund_inr), 2)
    }

    with open(REFUND_LEDGER, "a", encoding="utf-8") as file:
        file.write(json.dumps(entry) + "\n")

    return {"refund_inr": round(float(refund_inr), 2)}


def issue_refund(order_id, refund_inr):
    return record_refund_action(
        "issue_refund", order_id, refund_inr
    )


def escalate_to_human(order_id, refund_inr):
    return record_refund_action(
        "escalate_to_human", order_id, refund_inr
    )


TOOLS_SPEC = [
    {
        "name": "get_order",
        "description": "Look up an order in the Instamart database.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"}
            },
            "required": ["order_id"]
        }
    },
    {
        "name": "search_policy",
        "description": "Retrieve relevant policy document chunks.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string"}
            },
            "required": ["query"]
        }
    },
    {
        "name": "calculate_refund",
        "description": "Calculate the eligible late-delivery refund.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_amount": {"type": "number"},
                "delivery_minutes": {"type": "number"}
            },
            "required": ["order_amount", "delivery_minutes"]
        }
    },
    {
        "name": "issue_refund",
        "description": "Record an automatically approved refund.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"},
                "refund_inr": {"type": "number"}
            },
            "required": ["order_id", "refund_inr"]
        }
    },
    {
        "name": "escalate_to_human",
        "description": "Record a refund requiring human approval.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"},
                "refund_inr": {"type": "number"}
            },
            "required": ["order_id", "refund_inr"]
        }
    }
]


TOOL_MAP = {
    "get_order": get_order,
    "search_policy": search_policy,
    "calculate_refund": calculate_refund,
    "issue_refund": issue_refund,
    "escalate_to_human": escalate_to_human
}


print("Database exists:", DATABASE_PATH.exists())
print("Tools defined:", list(TOOL_MAP.keys()))
print("Tool specifications:", len(TOOLS_SPEC))

Database exists: True
Tools defined: ['get_order', 'search_policy', 'calculate_refund', 'issue_refund', 'escalate_to_human']
Tool specifications: 5


## ReAct Agent and Checkpointing

The ReAct agent follows a Reasoning and Acting workflow:

1. Send the user's request and available tools to the mock LLM.
2. If the LLM requests a tool, execute it and capture its observation.
3. Append the tool observation to the conversation.
4. Save a checkpoint after every tool execution.
5. Continue until the LLM returns a final answer or a safety limit is reached.

The agent also supports resuming from a saved checkpoint. It enforces limits on model calls, token usage, and execution time. If a limit is reached, it returns a fallback status and a human handoff message.

In [12]:
class SimulatedCrash(Exception):
    """Used to demonstrate recovery from a saved checkpoint."""
    pass


def checkpoint_path(run_id):
    return STATE_DIR / f"{run_id}.json"


def save_checkpoint(state):
    path = checkpoint_path(state["run_id"])
    with open(path, "w", encoding="utf-8") as file:
        json.dump(state, file, indent=2, default=str)


def agent_fallback(status, state, reason):
    return {
        "status": status,
        "final_answer": f"{status}: {reason} Please hand this request to a human support agent.",
        "steps": state["steps"],
        "llm_calls": state["llm_calls"],
        "tokens_used": state["tokens_used"],
        "run_id": state["run_id"]
    }


def continue_agent(state, crash_after_tool=None):
    start_time = time.monotonic()
    tools_this_call = 0

    while state["llm_calls"] < state["max_iterations"]:

        # Check the time limit before making another model call.
        if time.monotonic() - start_time >= state["time_limit_seconds"]:
            return agent_fallback(
                "FALLBACK_TIMEOUT", state,
                "The agent reached its execution time limit."
            )

        response = mock_llm.chat(state["messages"], tools=TOOLS_SPEC)
        state["llm_calls"] += 1

        call_tokens = response.usage.total_tokens
        state["tokens_used"] += call_tokens

        # Stop before executing a requested tool if the token limit is exceeded.
        if state["tokens_used"] > state["token_limit"]:
            return agent_fallback(
                "FALLBACK_TOKEN_LIMIT", state,
                "The agent exceeded its token limit."
            )

        message = response.choices[0].message
        tool_calls = message.tool_calls

        # No tool call means the model has returned its final answer.
        if not tool_calls:
            final_answer = message.content or "No final answer was returned."
            status = final_answer.split(":", 1)[0].strip().upper()

            return {
                "status": status,
                "final_answer": final_answer,
                "steps": state["steps"],
                "llm_calls": state["llm_calls"],
                "tokens_used": state["tokens_used"],
                "run_id": state["run_id"]
            }

        # Record the assistant's requested tool calls.
        assistant_tool_calls = []
        for tool_call in tool_calls:
            arguments = tool_call.arguments
            if isinstance(arguments, str):
                arguments = json.loads(arguments)

            assistant_tool_calls.append({
                "id": tool_call.id,
                "name": tool_call.name,
                "arguments": arguments
            })

        state["messages"].append({
            "role": "assistant",
            "content": None,
            "tool_calls": assistant_tool_calls
        })

        # Execute each requested tool and save its observation.
        for tool_call in assistant_tool_calls:
            tool_name = tool_call["name"]
            arguments = tool_call["arguments"]

            if tool_name not in TOOL_MAP:
                observation = {"error": f"Unknown tool: {tool_name}"}
            else:
                observation = TOOL_MAP[tool_name](**arguments)

            state["messages"].append({
                "role": "tool",
                "tool_call_id": tool_call["id"],
                "name": tool_name,
                "content": json.dumps(observation)
            })

            state["steps"].append({
                "tool": tool_name,
                "arguments": arguments,
                "observation": observation
            })

            state["step_number"] += 1
            state["tools_executed"].append(tool_name)
            tools_this_call += 1

            # Save after every tool observation, before any possible crash.
            save_checkpoint(state)

            if crash_after_tool is not None and tools_this_call >= crash_after_tool:
                raise SimulatedCrash(
                    f"Simulated crash after tool execution {tools_this_call}."
                )

    return agent_fallback(
        "FALLBACK_MAX_ITERATIONS", state,
        "The agent reached its maximum number of model calls without a final answer."
    )


def run_agent(
    user_message,
    scenario_id,
    max_iterations=6,
    token_limit=6000,
    time_limit_seconds=10,
    run_id=None,
    crash_after_tool=None
):
    if run_id is None:
        run_id = f"{scenario_id}_{time.time_ns()}"

    state = {
        "run_id": run_id,
        "scenario_id": scenario_id,
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are an AGENT for the Instamart refund workflow. "
                    "Use the available tools to check the order and relevant policy. "
                    "Follow the refund rules strictly. Only completed orders are eligible. "
                    "Refunds above INR 1000 must be escalated to a human. "
                    "Do not claim that a refund was issued unless the issue_refund tool "
                    "confirms it. Return a concise final answer beginning with "
                    "REFUND, ESCALATE, DENIED, or NOT_FOUND as appropriate."
                )
            },
            {"role": "user", "content": user_message}
        ],
        "steps": [],
        "tokens_used": 0,
        "llm_calls": 0,
        "step_number": 0,
        "tools_executed": [],
        "max_iterations": max_iterations,
        "token_limit": token_limit,
        "time_limit_seconds": time_limit_seconds
    }

    save_checkpoint(state)
    return continue_agent(state, crash_after_tool=crash_after_tool)


def resume_agent(run_id):
    path = checkpoint_path(run_id)

    if not path.exists():
        raise FileNotFoundError(f"No checkpoint found for run: {run_id}")

    with open(path, "r", encoding="utf-8") as file:
        state = json.load(file)

    return continue_agent(state)


print("ReAct agent and checkpoint functions defined.")
print("Checkpoint directory:", STATE_DIR)

ReAct agent and checkpoint functions defined.
Checkpoint directory: d:\Capstone Project\Intelligence & Support Copilot — Capstone Project\Intelligence & Support Copilot\state


## Test Refund Tools

Before running the complete agent, test the read-only tools and refund calculation independently.

This step checks:
- Whether an order can be retrieved from SQLite.
- Whether the policy search returns relevant chunks.
- Whether the refund calculation applies the 45-minute rule correctly.

We will not call `issue_refund` or `escalate_to_human` in this test, so the refund ledger remains untouched.

In [13]:
# Inspect the scenario file to identify a valid order ID.
print("Scenario columns:", agent_scenarios.columns.tolist())
display(agent_scenarios.head())

# Test policy retrieval.
policy_test = search_policy("refund policy late delivery")
print("\nPolicy search sources:")
for item in policy_test["results"]:
    print(item["source"], round(item["score"], 3))

# Test refund calculation.
print("\nRefund calculation tests:")
print("Delivery in 60 minutes:", calculate_refund(500, 60))
print("Delivery in 45 minutes:", calculate_refund(500, 45))
print("Delivery in 30 minutes:", calculate_refund(500, 30))

Scenario columns: ['scenario_id', 'customer_id', 'message', 'expected_outcome', 'expected_refund_inr']


,scenario_id,customer_id,message,expected_outcome,expected_refund_inr
0,S1,C0007,My order ORD-100048 reached me very late. I wo...,REFUND,128.25
1,S2,C0008,My order ORD-100067 reached me very late. I wo...,REFUND,135.00
2,S3,C0016,"Order ORD-100145 arrived far too late, please ...",ESCALATE,1999.00
3,S4,C0001,"Please refund order ORD-100001, I did not like...",DENIED,0.00
4,S5,C0003,I want my money back for order ORD-100025.,DENIED,0.00



Policy search sources:
refund_policy.txt 0.272
refund_policy.txt 0.251
refund_policy.txt 0.245

Refund calculation tests:
Delivery in 60 minutes: {'refund_inr': 500.0}
Delivery in 45 minutes: {'refund_inr': 0.0}
Delivery in 30 minutes: {'refund_inr': 0.0}


## Verify Scenario Order Records

Retrieve the database records for scenarios S1, S2 and S3.

These scenarios cover:
- S1 and S2: refunds below INR 1000.
- S3: refund above INR 1000, requiring human escalation.

Verify the order status, amount and delivery time against the expected scenario outcomes.

In [14]:
for scenario_id in ["S1", "S2", "S3"]:
    scenario = agent_scenarios.loc[
        agent_scenarios["scenario_id"] == scenario_id
    ].iloc[0]

    order_match = re.search(r"ORD-\d{6}", scenario["message"])
    order_id = order_match.group(0)

    print(f"\n{scenario_id} | {order_id}")
    print("Expected outcome:", scenario["expected_outcome"])
    print("Expected refund:", scenario["expected_refund_inr"])
    print("Database record:", get_order(order_id))


S1 | ORD-100048
Expected outcome: REFUND
Expected refund: 128.25
Database record: {'order_id': 'ORD-100048', 'status': 'completed', 'order_amount': 128.25, 'delivery_minutes': 65.0}

S2 | ORD-100067
Expected outcome: REFUND
Expected refund: 135.0
Database record: {'order_id': 'ORD-100067', 'status': 'completed', 'order_amount': 135.0, 'delivery_minutes': 59.0}

S3 | ORD-100145
Expected outcome: ESCALATE
Expected refund: 1999.0
Database record: {'order_id': 'ORD-100145', 'status': 'completed', 'order_amount': 1999.0, 'delivery_minutes': 54.0}


## Run a Complete Refund Agent Test

Run the ReAct agent for scenario S2.

The agent should:
1. Extract the order ID from the customer's message.
2. Retrieve the order from SQLite.
3. Search the refund policy.
4. Calculate the eligible refund.
5. Issue the refund because it is below INR 1000.
6. Return a final REFUND decision.

The tool execution and observations will be saved in a checkpoint.

In [15]:
s2 = agent_scenarios.loc[
    agent_scenarios["scenario_id"] == "S2"
].iloc[0]

s2_result = run_agent(
    user_message=s2["message"],
    scenario_id="S2"
)

print("Status:", s2_result["status"])
print("Final answer:", s2_result["final_answer"])
print("Model calls:", s2_result["llm_calls"])
print("Tokens used:", s2_result["tokens_used"])
print("\nTool execution sequence:")

for step in s2_result["steps"]:
    print(step["tool"], "->", step["observation"])

Status: REFUND
Final answer: REFUND: INR 135.0 issued.
Model calls: 5
Tokens used: 1277

Tool execution sequence:
get_order -> {'order_id': 'ORD-100067', 'status': 'completed', 'order_amount': 135.0, 'delivery_minutes': 59.0}
search_policy -> {'results': [{'source': 'refund_policy.txt', 'score': 0.2719281507693355, 'text': 'order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are'}, {'source': 'refund_policy.txt', 'score': 0.25137413448459545, 'text': 'Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer'}, {'source': 'refund_policy.txt', 'score': 0.24540580878362586, 'text': 'late-delivery refund is paid. Refunds up to INR 1000 are approved automat

## Crash-and-Resume Demonstration

Scenario S1 is used to demonstrate checkpoint recovery.

The agent will:
1. Start processing S1.
2. Execute `get_order`.
3. Execute `search_policy`.
4. Save the checkpoint after the second tool observation.
5. Simulate a crash immediately after that checkpoint.
6. Resume from the saved checkpoint using `resume_agent()`.

The resumed agent should continue from the next step without repeating either completed tool. It should finish with a refund of INR 128.25.

In [16]:
s1 = agent_scenarios.loc[
    agent_scenarios["scenario_id"] == "S1"
].iloc[0]

crash_run_id = "S1_crash_resume_demo"

# Start the run and simulate a crash after the second tool execution.
try:
    run_agent(
        user_message=s1["message"],
        scenario_id="S1",
        run_id=crash_run_id,
        crash_after_tool=2
    )
except SimulatedCrash as error:
    print(error)

# Inspect the saved checkpoint before resuming.
with open(checkpoint_path(crash_run_id), "r", encoding="utf-8") as file:
    saved_state = json.load(file)

print("\nCheckpoint saved:")
print("Scenario:", saved_state["scenario_id"])
print("Completed tools:", saved_state["tools_executed"])
print("Saved steps:", saved_state["step_number"])
print("Model calls so far:", saved_state["llm_calls"])

# Resume the same run.
s1_resumed = resume_agent(crash_run_id)

print("\nResumed result:")
print("Status:", s1_resumed["status"])
print("Final answer:", s1_resumed["final_answer"])
print("Model calls:", s1_resumed["llm_calls"])
print("Tokens used:", s1_resumed["tokens_used"])
print("Complete tool sequence:")

for step in s1_resumed["steps"]:
    print(step["tool"], "->", step["observation"])

Simulated crash after tool execution 2.

Checkpoint saved:
Scenario: S1
Completed tools: ['get_order', 'search_policy']
Saved steps: 2
Model calls so far: 2

Resumed result:
Status: REFUND
Final answer: REFUND: INR 128.25 issued.
Model calls: 5
Tokens used: 1277
Complete tool sequence:
get_order -> {'order_id': 'ORD-100048', 'status': 'completed', 'order_amount': 128.25, 'delivery_minutes': 65.0}
search_policy -> {'results': [{'source': 'refund_policy.txt', 'score': 0.2719281507693355, 'text': 'order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are'}, {'source': 'refund_policy.txt', 'score': 0.25137413448459545, 'text': 'Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more tha

## Test Agent Safety Limits

Test two fallback conditions:

1. Token limit: set the token limit to 200. If the first model call exceeds this limit, the agent must stop before executing the requested tool.
2. Time limit: set the time limit to zero seconds. The agent must stop before making its first model call.

Both cases should return a human handoff message and must not issue a refund.

In [17]:
# Test token limit.
token_demo = run_agent(
    user_message=s1["message"],
    scenario_id="S1_TOKEN_DEMO",
    token_limit=200,
    run_id="S1_token_limit_demo"
)

print("TOKEN LIMIT DEMO")
print("Status:", token_demo["status"])
print("Model calls:", token_demo["llm_calls"])
print("Tokens used:", token_demo["tokens_used"])
print("Tools executed:", [step["tool"] for step in token_demo["steps"]])
print("Final answer:", token_demo["final_answer"])

# Test time limit.
time_demo = run_agent(
    user_message=s1["message"],
    scenario_id="S1_TIMEOUT_DEMO",
    time_limit_seconds=0,
    run_id="S1_timeout_demo"
)

print("\nTIME LIMIT DEMO")
print("Status:", time_demo["status"])
print("Model calls:", time_demo["llm_calls"])
print("Tokens used:", time_demo["tokens_used"])
print("Tools executed:", [step["tool"] for step in time_demo["steps"]])
print("Final answer:", time_demo["final_answer"])

TOKEN LIMIT DEMO
Status: FALLBACK_TOKEN_LIMIT
Model calls: 2
Tokens used: 254
Tools executed: ['get_order']
Final answer: FALLBACK_TOKEN_LIMIT: The agent exceeded its token limit. Please hand this request to a human support agent.

TIME LIMIT DEMO
Status: FALLBACK_TIMEOUT
Model calls: 0
Tokens used: 0
Tools executed: []
Final answer: FALLBACK_TIMEOUT: The agent reached its execution time limit. Please hand this request to a human support agent.


## Execute the Seven Refund Agent Scenarios

Run the ReAct agent against all seven scenarios from agent_scenarios.csv.

For each scenario, record:
- Scenario ID
- Final status and answer
- Model calls and tokens used
- Every executed tool, its arguments and observation

Save exactly one JSON record per scenario in outputs/agent_runs.jsonl.

The expected outcomes include refunds, escalation, denial, not-found handling, and a maximum-iteration fallback.

In [18]:
AGENT_RUNS_FILE = OUTPUTS / "agent_runs.jsonl"

# Start with a clean results file so it contains exactly seven scenario runs.
if AGENT_RUNS_FILE.exists():
    AGENT_RUNS_FILE.unlink()

scenario_results = []

for _, scenario in agent_scenarios.iterrows():
    result = run_agent(
        user_message=scenario["message"],
        scenario_id=scenario["scenario_id"]
    )

    record = {
        "scenario_id": scenario["scenario_id"],
        "status": result["status"],
        "final_answer": result["final_answer"],
        "llm_calls": result["llm_calls"],
        "tokens_used": result["tokens_used"],
        "steps": result["steps"]
    }

    scenario_results.append(record)

    with open(AGENT_RUNS_FILE, "a", encoding="utf-8") as file:
        file.write(json.dumps(record) + "\n")

    print(
        scenario["scenario_id"],
        "| Expected:", scenario["expected_outcome"],
        "| Actual:", result["status"],
        "| Calls:", result["llm_calls"],
        "| Tokens:", result["tokens_used"]
    )

print("\nSaved agent runs:", AGENT_RUNS_FILE)
print("Scenario count:", len(scenario_results))

S1 | Expected: REFUND | Actual: REFUND | Calls: 5 | Tokens: 1277
S2 | Expected: REFUND | Actual: REFUND | Calls: 5 | Tokens: 1277
S3 | Expected: ESCALATE | Actual: ESCALATE | Calls: 5 | Tokens: 1264
S4 | Expected: DENIED | Actual: DENIED | Calls: 4 | Tokens: 909
S5 | Expected: DENIED | Actual: DENIED | Calls: 3 | Tokens: 565
S6 | Expected: NOT_FOUND | Actual: NOT_FOUND | Calls: 2 | Tokens: 224
S7 | Expected: FALLBACK | Actual: FALLBACK_MAX_ITERATIONS | Calls: 6 | Tokens: 3456

Saved agent runs: d:\Capstone Project\Intelligence & Support Copilot — Capstone Project\Intelligence & Support Copilot\outputs\agent_runs.jsonl
Scenario count: 7


## Validate Agent Outputs and Refund Ledger

Check the seven scenario results against their expected outcomes and refund amounts.

Also verify:
- The agent_runs.jsonl file contains exactly seven valid JSON records.
- The refund ledger contains valid JSON records.
- Refund and escalation actions use the correct tool names.
- S1, S2 and S3 have the expected refund or escalation amounts.
- Denied, not-found and fallback scenarios do not issue refunds.

Note: The refund ledger may also contain entries from the earlier standalone S2 test and S1 crash-and-resume demonstration.

In [19]:
# Build a concise summary from the seven batch results.
summary_rows = []

for _, scenario in agent_scenarios.iterrows():
    result = next(
        r for r in scenario_results
        if r["scenario_id"] == scenario["scenario_id"]
    )

    action_steps = [
        step for step in result["steps"]
        if step["tool"] in ["issue_refund", "escalate_to_human"]
    ]

    actual_amount = (
        action_steps[-1]["observation"]["refund_inr"]
        if action_steps else 0.0
    )

    expected_status = scenario["expected_outcome"]
    actual_status = result["status"]

    status_ok = (
        actual_status == expected_status
        or (
            expected_status == "FALLBACK"
            and actual_status.startswith("FALLBACK_")
        )
    )

    amount_ok = abs(
        float(actual_amount) - float(scenario["expected_refund_inr"])
    ) < 0.01

    summary_rows.append({
        "scenario_id": scenario["scenario_id"],
        "expected": expected_status,
        "actual": actual_status,
        "expected_refund_inr": float(scenario["expected_refund_inr"]),
        "actual_refund_inr": float(actual_amount),
        "status_pass": status_ok,
        "amount_pass": amount_ok,
        "llm_calls": result["llm_calls"],
        "tokens_used": result["tokens_used"]
    })

agent_summary = pd.DataFrame(summary_rows)
display(agent_summary)

# Validate the seven-line agent run file.
with open(AGENT_RUNS_FILE, "r", encoding="utf-8") as file:
    agent_run_lines = [json.loads(line) for line in file if line.strip()]

print("Agent run records:", len(agent_run_lines))
print("All seven records valid:", len(agent_run_lines) == 7)

# Validate the refund ledger.
ledger_records = []

if REFUND_LEDGER.exists():
    with open(REFUND_LEDGER, "r", encoding="utf-8") as file:
        ledger_records = [
            json.loads(line) for line in file if line.strip()
        ]

print("\nRefund ledger records:", len(ledger_records))
print(
    "Allowed ledger tools:",
    all(
        record["tool"] in ["issue_refund", "escalate_to_human"]
        for record in ledger_records
    )
)

print("\nBatch actions:")
for scenario_id in ["S1", "S2", "S3"]:
    result = next(
        r for r in scenario_results
        if r["scenario_id"] == scenario_id
    )
    actions = [
        (step["tool"], step["observation"])
        for step in result["steps"]
        if step["tool"] in ["issue_refund", "escalate_to_human"]
    ]
    print(scenario_id, actions)

all_passed = (
    agent_summary["status_pass"].all()
    and agent_summary["amount_pass"].all()
    and len(agent_run_lines) == 7
    and all(
        record["tool"] in ["issue_refund", "escalate_to_human"]
        for record in ledger_records
    )
)

print("\nPART D AGENT OUTPUT CHECK:", "PASSED" if all_passed else "FAILED")

,scenario_id,expected,actual,expected_refund_inr,actual_refund_inr,status_pass,amount_pass,llm_calls,tokens_used
0,S1,REFUND,REFUND,128.25,128.25,True,True,5,1277
1,S2,REFUND,REFUND,135.00,135.00,True,True,5,1277
2,S3,ESCALATE,ESCALATE,1999.00,1999.00,True,True,5,1264
3,S4,DENIED,DENIED,0.00,0.00,True,True,4,909
4,S5,DENIED,DENIED,0.00,0.00,True,True,3,565
5,S6,NOT_FOUND,NOT_FOUND,0.00,0.00,True,True,2,224
6,S7,FALLBACK,FALLBACK_MAX_ITERATIONS,0.00,0.00,True,True,6,3456


Agent run records: 7
All seven records valid: True

Refund ledger records: 5
Allowed ledger tools: True

Batch actions:
S1 [('issue_refund', {'refund_inr': 128.25})]
S2 [('issue_refund', {'refund_inr': 135.0})]
S3 [('escalate_to_human', {'refund_inr': 1999.0})]

PART D AGENT OUTPUT CHECK: PASSED


## ReAct vs Plan-and-Execute

### ReAct Approach

ReAct combines reasoning and tool execution. The agent observes each tool result and decides what to do next. This makes it suitable for workflows where decisions depend on database records or policy results.

### Plan-and-Execute Approach

Plan-and-Execute creates a sequence of steps before execution and then follows that plan. It is useful when the workflow is predictable and the required actions are known in advance.

### Measured ReAct Performance

| Scenario | Model calls | Tokens used |
|---|---:|---:|
| S1 – Refund | 5 | 1,277 |
| S5 – Denied | 3 | 565 |
| S6 – Order not found | 2 | 224 |

The agent uses fewer calls when it can reach a decision early. S6 requires only two calls because the order does not exist.

### Handling Unexpected Results

ReAct can use an unexpected tool observation to change its next action. For example, if an order is not found or its status is cancelled, the agent can stop the refund workflow and return an appropriate response. Plan-and-Execute may require additional checks or replanning when an observation differs from its original assumptions.

### Conclusion

ReAct is the better choice for this refund workflow because the next action depends on the order status, delivery time, refund amount and policy rules. It provides flexible decision-making while keeping tool execution and observations traceable.

## Part D Final Acceptance Checklist

Verify that all required Part D deliverables are present and working.

The checks cover:
- 15 policy chunks with 40-word chunks and 10-word overlap.
- 14 policy evaluation questions.
- Seven agent scenarios.
- Correct scenario decisions and refund amounts.
- Crash-and-resume checkpoint.
- Token, timeout and maximum-iteration fallbacks.
- Agent run records and refund ledger.
- Required output files.

This is the final validation before integrating the project.

In [20]:
checks = {}

# Policy RAG
checks["15 policy chunks"] = len(chunks_df) == 15
checks["14 policy evaluation questions"] = len(policy_eval) == 14
checks["TF-IDF index created"] = chunk_vectors.shape[0] == 15

expected_chunk_counts = {
    "delivery_policy.txt": 3,
    "escalation_policy.txt": 3,
    "payment_policy.txt": 3,
    "privacy_policy.txt": 2,
    "refund_policy.txt": 4
}
actual_chunk_counts = chunks_df["source"].value_counts().to_dict()
checks["Policy chunk distribution"] = actual_chunk_counts == expected_chunk_counts

# Agent scenarios
checks["Seven scenarios executed"] = len(scenario_results) == 7
checks["All scenario decisions passed"] = agent_summary["status_pass"].all()
checks["All refund amounts passed"] = agent_summary["amount_pass"].all()

# Crash recovery and safety limits
checks["Crash recovery passed"] = (
    s1_resumed["status"] == "REFUND"
    and s1_resumed["llm_calls"] == 5
    and s1_resumed["steps"][0]["tool"] == "get_order"
    and s1_resumed["steps"][1]["tool"] == "search_policy"
)
checks["Token limit fallback passed"] = (
    token_demo["status"] == "FALLBACK_TOKEN_LIMIT"
)
checks["Timeout fallback passed"] = (
    time_demo["status"] == "FALLBACK_TIMEOUT"
)
checks["Maximum iteration fallback passed"] = (
    next(r for r in scenario_results if r["scenario_id"] == "S7")["status"]
    == "FALLBACK_MAX_ITERATIONS"
)

# Saved files
checks["Agent runs file exists"] = AGENT_RUNS_FILE.exists()
checks["Refund ledger exists"] = REFUND_LEDGER.exists()
checks["Crash checkpoint exists"] = checkpoint_path(crash_run_id).exists()

# Check agent run JSONL count
with open(AGENT_RUNS_FILE, "r", encoding="utf-8") as file:
    saved_agent_runs = [json.loads(line) for line in file if line.strip()]

checks["Exactly seven saved agent records"] = len(saved_agent_runs) == 7

# Display results
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} - {name}")

print("\nPart D acceptance:", "ALL PASSED" if all(checks.values()) else "CHECK FAILURES")

PASS - 15 policy chunks
PASS - 14 policy evaluation questions
PASS - TF-IDF index created
PASS - Policy chunk distribution
PASS - Seven scenarios executed
PASS - All scenario decisions passed
PASS - All refund amounts passed
PASS - Crash recovery passed
PASS - Token limit fallback passed
PASS - Timeout fallback passed
PASS - Maximum iteration fallback passed
PASS - Agent runs file exists
PASS - Refund ledger exists
PASS - Crash checkpoint exists
PASS - Exactly seven saved agent records

Part D acceptance: ALL PASSED


In [22]:
import ast

SRC_DIR = PROJECT_ROOT / "src"

print("PROJECT FILE INVENTORY")
print("=" * 60)

for path in sorted(PROJECT_ROOT.iterdir()):
    if path.is_dir():
        print(f"[DIR]  {path.name}/")
    else:
        print(f"[FILE] {path.name}")

print("\nSOURCE MODULES")
print("=" * 60)

for path in sorted(SRC_DIR.glob("*.py")):
    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    print(f"\n{path.name} ({len(source)} characters)")

    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            print(f"  Function: {node.name}")
        elif isinstance(node, ast.ClassDef):
            print(f"  Class: {node.name}")
        elif isinstance(node, (ast.Import, ast.ImportFrom)):
            print(f"  Import: {ast.unparse(node)}")

print("\nENTRY POINT")
print("=" * 60)

entry_file = PROJECT_ROOT / "run_project.py"
if entry_file.exists():
    print(entry_file.read_text(encoding="utf-8"))
else:
    print("run_project.py does not exist.")

print("\nREQUIREMENTS")
print("=" * 60)

requirements_file = PROJECT_ROOT / "requirements.txt"
if requirements_file.exists():
    print(requirements_file.read_text(encoding="utf-8"))
else:
    print("requirements.txt does not exist.")

print("\nREADME HEADINGS")
print("=" * 60)

readme_file = PROJECT_ROOT / "README.md"
if readme_file.exists():
    for line in readme_file.read_text(encoding="utf-8").splitlines():
        if line.startswith("#"):
            print(line)
else:
    print("README.md does not exist.")

PROJECT FILE INVENTORY
[DIR]  .git/
[FILE] .gitignore
[DIR]  .venv/
[DIR]  __pycache__/
[DIR]  data/
[FILE] generate_data.py
[FILE] GITHUB_RESOURCES.md
[FILE] mock_llm.py
[DIR]  notebooks/
[DIR]  outputs/
[FILE] README.md
[FILE] requirements.txt
[FILE] run_project.py
[DIR]  sql/
[DIR]  src/
[DIR]  state/

SOURCE MODULES

__init__.py (0 characters)

customer_models.py (6742 characters)
  Import: from pathlib import Path
  Import: import numpy as np, pandas as pd, joblib
  Import: from sklearn.cluster import KMeans
  Import: from sklearn.preprocessing import StandardScaler, OneHotEncoder
  Import: from sklearn.compose import ColumnTransformer
  Import: from sklearn.pipeline import Pipeline
  Import: from sklearn.model_selection import train_test_split, GridSearchCV
  Import: from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, accuracy_score
  Import: from sklearn.linear_model import LogisticRegression
  Import: from sklearn.ensemble import RandomForestC